In [1]:
import json
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_PATH = "../data/raw/movies_raw_enriched.csv"

movies_df = pd.read_csv(DATA_PATH)

In [3]:
movies_df.head(2)

,movie_id,title,overview,release_date,genres,keywords,runtime,vote_average,vote_count,poster_url,cast,director
0,11,Star Wars,Princess Leia is captured and held hostage by ...,1977-05-25,"[""Adventure"", ""Action"", ""Science Fiction""]","[""empire"", ""galaxy"", ""rebellion"", ""android"", ""...",121,8.208,23043,https://image.tmdb.org/t/p/w500fai0rspsNeJCS69...,Mark Hamill | Harrison Ford | Carrie Fisher | ...,George Lucas
1,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",2003-05-30,"[""Animation"", ""Family"", ""Adventure""]","[""fish"", ""sydney, australia"", ""parent child re...",100,7.822,21126,https://image.tmdb.org/t/p/w500eHuGQ10FUzK1mdO...,Albert Brooks | Ellen DeGeneres | Alexander Go...,Andrew Stanton


In [4]:
movies_df.shape

(35001, 12)

In [5]:
movies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35001 entries, 0 to 35000
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movie_id      35001 non-null  int64  
 1   title         35001 non-null  str    
 2   overview      31659 non-null  str    
 3   release_date  34959 non-null  str    
 4   genres        35001 non-null  str    
 5   keywords      35001 non-null  str    
 6   runtime       35001 non-null  int64  
 7   vote_average  35001 non-null  float64
 8   vote_count    35001 non-null  int64  
 9   poster_url    31452 non-null  str    
 10  cast          33343 non-null  str    
 11  director      33994 non-null  str    
dtypes: float64(1), int64(3), str(8)
memory usage: 18.8 MB


In [6]:
movies = movies_df[
    [
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "director"
    ]
].copy()

In [7]:
movies.isnull().sum()

movie_id       0
title          0
overview    3342
genres         0
keywords       0
cast        1658
director    1007
dtype: int64

In [8]:
movies = movies.dropna(
    subset=[
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "director"
    ]
).reset_index(drop=True)

movies.shape

(29820, 7)

In [9]:
exact_duplicates = movies.duplicated().sum()

duplicate_ids = movies["movie_id"].duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicates:,}")

print(f"Duplicate movie IDs: {duplicate_ids:,}")

Exact duplicate rows: 0
Duplicate movie IDs: 0


In [10]:
movies["title"] = movies["title"].str.strip()

movies = movies[
    movies["title"] != ""
].reset_index(drop=True)

movies.shape

(29820, 7)

In [11]:
def convert(obj):
    try:
        result = json.loads(obj)

        if isinstance(result, list):
            return result

        return []

    except (json.JSONDecodeError, TypeError):
        return []

In [12]:
movies["genres"] = movies["genres"].apply(convert)

movies["keywords"] = movies["keywords"].apply(convert)

movies["genres"] = movies["genres"].apply(
    lambda x: " ".join(x)
)

movies["keywords"] = movies["keywords"].apply(
    lambda x: " ".join(x)
)

movies["cast"] = movies["cast"].str.replace(
    "|", " ", regex=False
)

movies["director"] = movies["director"].str.replace(
    "|", " ", regex=False
)

In [13]:
def build_text(row):
    return " ".join([
        row["overview"],
        row["genres"],
        row["keywords"],
        row["cast"],
        row["director"]
    ])

In [14]:
movies["text"] = movies.apply(build_text, axis=1)

In [15]:
new_df = movies[["movie_id", "title", "text"]].copy()

In [16]:
new_df["text"] = new_df["text"].str.lower()

In [17]:
print("Final dataset shape:", new_df.shape)
print("Total movies:", len(new_df))

Final dataset shape: (29820, 3)
Total movies: 29820


In [18]:
new_df.head()

,movie_id,title,text
0,11,Star Wars,princess leia is captured and held hostage by ...
1,12,Finding Nemo,"nemo, an adventurous young clownfish, is unexp..."
2,13,Forrest Gump,a man with a low iq has accomplished great thi...
3,14,American Beauty,"lester burnham, a depressed suburban father in..."
4,15,Citizen Kane,newspaper magnate charles foster kane is taken...


In [19]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(new_df["text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (29820, 103519)


In [20]:
def recommend(title, movie_id=None, n=5):

    matches = new_df.index[
        new_df["title"].str.casefold()
        == title.strip().casefold()
    ].tolist()

    if not matches:
        return f"Movie not found: {title}"

    if movie_id is not None:
        matches = [
            i for i in matches
            if new_df.iloc[i]["movie_id"] == movie_id
        ]

        if not matches:
            return f"Movie ID {movie_id} does not match '{title}'."

    movie_index = matches[0]

    selected_title = (
        new_df.iloc[movie_index]["title"].strip().casefold()
    )

    distances = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    ranked_indices = distances.argsort()[::-1]

    ranked_indices = [
        i for i in ranked_indices
        if (
            i != movie_index
            and new_df.iloc[i]["title"].strip().casefold()
            != selected_title
        )
    ][:max(0, n)]

    results = new_df.iloc[ranked_indices][
        ["movie_id", "title"]
    ].copy()

    results["similarity"] = distances[ranked_indices]

    return results.reset_index(drop=True)

In [24]:
recommend("Iron Man")

,movie_id,title,similarity
0,10138,Iron Man 2,0.373337
1,68721,Iron Man 3,0.367134
2,315635,Spider-Man: Homecoming,0.313987
3,429617,Spider-Man: Far From Home,0.233730
4,99861,Avengers: Age of Ultron,0.222892


In [22]:
test_movies = [
    # Superhero / Action
    "The Dark Knight",
    "Iron Man",
    "Spider-Man",

    # Science fiction
    "Interstellar",
    "Inception",
    "The Matrix",

    # Romance
    "The Notebook",
    "Titanic",
    "La La Land",

    # Comedy
    "3 Idiots",
    "Home Alone",
    "The Hangover",

    # Animation
    "Toy Story",
    "Finding Nemo",
    "Coco",

    # Crime / Thriller
    "The Godfather",
    "Se7en",
    "Gone Girl",

    # Horror
    "The Conjuring",
    "A Nightmare on Elm Street",

    # Indian cinema
    "Drishyam",
    "Dangal",
    "3 Idiots",

    # Fantasy / Adventure
    "The Lord of the Rings",
    "Harry Potter and the Philosopher's Stone",

    # Mystery / Psychological
    "Shutter Island",
    "The Prestige",
]

for movie in test_movies:
    print(f"\n{'=' * 60}")
    print(f"Recommendations for: {movie}")
    print("=" * 60)

    result = recommend(movie)

    if isinstance(result, str):
        print(result)
    else:
        print(result.to_string(index=False))


Recommendations for: The Dark Knight
 movie_id                 title  similarity
    49026 The Dark Knight Rises    0.364445
      272         Batman Begins    0.324330
      268                Batman    0.307106
      414        Batman Forever    0.254688
   414906            The Batman    0.200988

Recommendations for: Iron Man
 movie_id                     title  similarity
    10138                Iron Man 2    0.373337
    68721                Iron Man 3    0.367134
   315635    Spider-Man: Homecoming    0.313987
   429617 Spider-Man: Far From Home    0.233730
    99861   Avengers: Age of Ultron    0.222892

Recommendations for: Spider-Man
 movie_id                               title  similarity
      559                        Spider-Man 3    0.445260
     1930              The Amazing Spider-Man    0.356050
   569094 Spider-Man: Across the Spider-Verse    0.319070
   969681           Spider-Man: Brand New Day    0.316921
   315635              Spider-Man: Homecoming    0.26523

KeyboardInterrupt: 

In [ ]:
import pickle
from pathlib import Path

# Save model artifacts
MODEL_DIR = Path("../backend")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_artifacts = {
    "movies": new_df.reset_index(drop=True),
    "tfidf_vectorizer": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "version": "v1"
}

model_path = MODEL_DIR / "movie_recommender.pkl"

with open(model_path, "wb") as file:
    pickle.dump(model_artifacts, file, protocol=pickle.HIGHEST_PROTOCOL)

print(f"V1 model saved to: {model_path.resolve()}")
print(f"Movies: {len(new_df):,}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"File size: {model_path.stat().st_size / (1024**2):.2f} MB")

V1 model saved to: D:\ML Project\Seenema\backend\movie_recommender.pkl
Movies: 29,820
TF-IDF matrix shape: (29820, 103519)
File size: 28.30 MB


In [ ]:
import pickle
from pathlib import Path

model_path = Path("../backend/movie_recommender.pkl")

with open(model_path, "rb") as file:
    saved_model = pickle.load(file)

loaded_movies = saved_model["movies"]
loaded_tfidf = saved_model["tfidf_vectorizer"]
loaded_matrix = saved_model["tfidf_matrix"]

print("Model version:", saved_model["version"])
print("Movies loaded:", len(loaded_movies))
print("TF-IDF matrix shape:", loaded_matrix.shape)
print("Vectorizer vocabulary size:", len(loaded_tfidf.vocabulary_))

assert loaded_matrix.shape[0] == len(loaded_movies)

print("\nV1 model saved and verified successfully!")

Model version: v1
Movies loaded: 29820
TF-IDF matrix shape: (29820, 103519)
Vectorizer vocabulary size: 103519

V1 model saved and verified successfully!
